## Input Required Information

If you do not already have a slice created, please either create one using your own method or via the GraphML Slice Builder notebook provided. Also remember to configure your local Jupyter enviornment if you have not already (can you reach out to the nodes?).

| Variable | Use |
| --- | --- |
| SLICE_NAME    | Name of the slice you wish to work on. |

In [1]:
SLICE_NAME = "Eibp_PTPLARGE8node"
CLIENT_PREFIX = "ipnode"

## Access the Slice

FabOrchestrator is a custom Python class that utilizes the existing FabLib API. Its purpose is to make repetitive code easier to manage and build around features that we previously used in our custom GENI orchestrator.

The orchestrator class is initalized here, which also means the slice and its nodes are now accessable as well..

In [2]:
from FabUtils_new import FabOrchestrator

try:
    manager = FabOrchestrator(SLICE_NAME)
    
except Exception as e:
    print(f"Exception: {e}")

User: tm3886@rit.edu bastion key is valid!
Configuration is valid
Slice name: Eibp_PTPLARGE8node
Slice and nodes were acquired successfully.


## Upload Configuration Script

Anything that would make sense to be configured that does not include package installation should be placed in the init_mtp.sh script. Currently, this modifies the tmux configuration after it is installed. 

In [3]:
configScript = "/home/fabric/work/EIBP/Scripts/remote_scripts/init_mtp.sh"
#/home/fabric/work/EIBP/Scripts/remote_scripts
manager.uploadFileParallel_dep(configScript, prefixList="C,D,A")

File to upload: /home/fabric/work/EIBP/Scripts/remote_scripts/init_mtp.sh
Placed in: /home/rocky/init_mtp.sh
Starting upload on node C1
Starting upload on node C2
Starting upload on node D1
Starting upload on node D2
Starting upload on node D3
Starting upload on node A1
Starting upload on node A2
Starting upload on node A3
Waiting for result from node C1
Output: -rw-rw-r--   1 1000     1000          292 20 Mar 16:41 ?
Waiting for result from node C2
Output: -rw-rw-r--   1 1000     1000          292 20 Mar 16:41 ?
Waiting for result from node D1
Output: -rw-rw-r--   1 1000     1000          292 20 Mar 16:41 ?
Waiting for result from node D2
Output: -rw-rw-r--   1 1000     1000          292 20 Mar 16:41 ?
Waiting for result from node D3
Output: -rw-rw-r--   1 1000     1000          292 20 Mar 16:41 ?
Waiting for result from node A1
Output: -rw-rw-r--   1 1000     1000          292 20 Mar 16:41 ?
Waiting for result from node A2
Output: -rw-rw-r--   1 1000     1000          292 20 Mar 16:4

## Upload stop time script

In [4]:
configStop = "/home/fabric/work/EIBP/Scripts/remote_scripts/stop_time.sh"
manager.uploadFileParallel_dep(configStop, prefixList="C,D,A")

File to upload: /home/fabric/work/EIBP/Scripts/remote_scripts/stop_time.sh
Placed in: /home/rocky/stop_time.sh
Starting upload on node C1
Starting upload on node C2
Starting upload on node D1
Starting upload on node D2
Starting upload on node D3
Starting upload on node A1
Starting upload on node A2
Starting upload on node A3
Waiting for result from node C1
Output: -rw-rw-r--   1 1000     1000           76 20 Mar 16:41 ?
Waiting for result from node C2
Output: -rw-rw-r--   1 1000     1000           76 20 Mar 16:41 ?
Waiting for result from node D1
Output: -rw-rw-r--   1 1000     1000           76 20 Mar 16:41 ?
Waiting for result from node D2
Output: -rw-rw-r--   1 1000     1000           76 20 Mar 16:41 ?
Waiting for result from node D3
Output: -rw-rw-r--   1 1000     1000           76 20 Mar 16:41 ?
Waiting for result from node A1
Output: -rw-rw-r--   1 1000     1000           76 20 Mar 16:41 ?
Waiting for result from node A2
Output: -rw-rw-r--   1 1000     1000           76 20 Mar 16

## Install Dependencies and Make Configuration Changes on all Nodes

Along with installing necessary pacakges, the initialization script uploaded in the prior cell is also run.

| Package | Use |
| --- | --- |
| tmux    | Terminal multiplexer similar to GNU Screen to allow for ssh sessions to disconnect and let the process continue to run. |
| Wireshark | Access to tshark, the command-line based version of the packet sniffer. |
| Development Tools | Includes all necessary applications, such as GNU GCC C compiler, make, debuggers, man pages. etc., which are needed to compile, build, and troubleshoot MTP code |

In [5]:
# The commands to run
packagesToInstall = "sudo dnf install -q -y tmux wireshark"
devToolsInstall = 'sudo dnf groupinstall -q -y "Development Tools"'
config = 'bash init_mtp.sh {name}'

# Execute the commands
manager.executeCommandsParallel_Dep(packagesToInstall)
manager.executeCommandsParallel_Dep(devToolsInstall)
manager.executeCommandsParallel_Dep(config, prefixList="C,D,A", addNodeName=True)

Starting command on node C1
Command to execute: sudo dnf install -q -y tmux wireshark
Starting command on node C2
Command to execute: sudo dnf install -q -y tmux wireshark
Starting command on node D1
Command to execute: sudo dnf install -q -y tmux wireshark
Starting command on node D2
Command to execute: sudo dnf install -q -y tmux wireshark
Starting command on node D3
Command to execute: sudo dnf install -q -y tmux wireshark
Starting command on node A1
Command to execute: sudo dnf install -q -y tmux wireshark
Starting command on node A2
Command to execute: sudo dnf install -q -y tmux wireshark
Starting command on node A3
Command to execute: sudo dnf install -q -y tmux wireshark
Starting command on node ipnode-1
Command to execute: sudo dnf install -q -y tmux wireshark
Starting command on node ipnode-2
Command to execute: sudo dnf install -q -y tmux wireshark
Starting command on node ipnode-3
Command to execute: sudo dnf install -q -y tmux wireshark

 Completed: 11/11 nodes succeeded
S

## Add Default Routes to the Compute Nodes

Sets up default routes for IP-based forwarding on compute nodes (prefix), allowing them to route traffic outside their local network. 

In [6]:
from ipaddress import ip_address, IPv4Address, IPv4Network
import re

SLICE_SUPERNET = "192.168.0.0/16"

for computeNode in manager.selectedNodes("ipnode"):
    # Get the compute node's interface
    for intf in computeNode.get_interfaces():
        ethName = intf.get_physical_os_interface_name()
        
        if("meas" not in ethName):
            # you can use ethName to do whatever you want example:
            # cmd = f"sudo ifconfig {ethName} down" 
            # Get the interface IPv4 address and its third octet
            ipAddress = intf.get_ip_addr()
            IPGroup = re.search(r"192\.168\.([0-9]{1,3})\.[0-9]{1,3}", ipAddress)
            thirdOctet = IPGroup.group(1)
            nextHop = f"192.168.{thirdOctet}.254"
       
    # Add the route to the node
    computeNode.ip_route_add(subnet=IPv4Network(SLICE_SUPERNET), gateway=IPv4Address(nextHop))
    
    print(f"Adding route {SLICE_SUPERNET} to {computeNode.get_name()} with next-hop {nextHop}")

Adding route 192.168.0.0/16 to ipnode-1 with next-hop 192.168.1.254
Adding route 192.168.0.0/16 to ipnode-2 with next-hop 192.168.2.254
Adding route 192.168.0.0/16 to ipnode-3 with next-hop 192.168.3.254


## Turn off Traditional IP-based Forwarding on EIBP Nodes

This is not done on compute/client nodes, as they are EIBP-unaware.

Disables traditional IP-based forwarding on EIBP nodes (prefixes) using sysctl commands.

In [7]:
cmdOff = "sudo sysctl -w net.ipv4.ip_forward=0"
cmdOn = "sudo sysctl -w net.ipv4.ip_forward=1"               
manager.executeCommandsParallel_Dep(cmdOff, prefixList="C,D,A")
manager.executeCommandsParallel_Dep(cmdOn, prefixList="ipnode")

Starting command on node C1
Command to execute: sudo sysctl -w net.ipv4.ip_forward=0
Starting command on node C2
Command to execute: sudo sysctl -w net.ipv4.ip_forward=0
Starting command on node D1
Command to execute: sudo sysctl -w net.ipv4.ip_forward=0
Starting command on node D2
Command to execute: sudo sysctl -w net.ipv4.ip_forward=0
Starting command on node D3
Command to execute: sudo sysctl -w net.ipv4.ip_forward=0
Starting command on node A1
Command to execute: sudo sysctl -w net.ipv4.ip_forward=0
Starting command on node A2
Command to execute: sudo sysctl -w net.ipv4.ip_forward=0
Starting command on node A3
Command to execute: sudo sysctl -w net.ipv4.ip_forward=0

 Completed: 8/8 nodes succeeded
Starting command on node ipnode-1
Command to execute: sudo sysctl -w net.ipv4.ip_forward=1
Starting command on node ipnode-2
Command to execute: sudo sysctl -w net.ipv4.ip_forward=1
Starting command on node ipnode-3
Command to execute: sudo sysctl -w net.ipv4.ip_forward=1

 Completed: 3